# Laboratório forense TSE 2026 — Fases S/T

**Primeiro turno presidencial de 2026.** Este notebook lê exclusivamente os resultados já produzidos pelos scripts do repositório (opções **S** e **T**). Não faz downloads, não reabre ZIP, não acessa SQLite e não altera os arquivos oficiais.

**Pergunta investigativa:** a composição geográfica dos BUs recebidos depois de **04/10/2026 20:04:39 BRT** explica a regularidade da curva pública? Quais desvios permanecem quando usamos apenas dados disponíveis antes do corte?

**Distinção essencial:** o horário de **publicação EA20** e o horário `DT_BU_RECEBIDO` do BWEB correspondem a etapas diferentes. As duas cronologias são analisadas separadamente; não se equiparam lotes pelos minutos. O modelo `MUNICIPIO_final_IN_SAMPLE` conhece dados futuros e serve apenas como referência retrospectiva.

## 1 — Preparação, caminhos e leitura com verificação SHA-256

O notebook identifica automaticamente o diretório do repositório e usa o mesmo caminho do disco **H:** definido em `urnas.root_path`. Não precisa editar caminhos no notebook. Se faltarem arquivos, execute **S** e **T** no menu `start_windows.bat`.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

def localizar_repo():
    for pasta in (Path.cwd(), *Path.cwd().parents):
        if (pasta / "start_windows.bat").is_file():
            return pasta
    raise FileNotFoundError("Abra o notebook a partir da pasta do repositorio TSE.")

REPO = localizar_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from urnas import root_path

ROOT = root_path(None)
BASE = ROOT / "analysis" / "periodicidade_bweb_2026"
CORTE = pd.Timestamp("2026-10-04 20:04:39")

def sha256(caminho):
    h = hashlib.sha256()
    with caminho.open("rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(4 * 1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def ler_json(nome):
    arquivo = BASE / nome
    if not arquivo.is_file():
        raise FileNotFoundError(f"Falta {arquivo}. Execute as opcoes S/T do menu.")
    return json.loads(arquivo.read_text(encoding="utf-8"))

s = ler_json("phase5o_operational_curve_2026_summary.json")
t = ler_json("phase5p_holdout_geographic_curve_2026_summary.json")
assert s["status"] == "CURVAS_OPERACIONAIS_RECONSTRUIDAS_2026"
assert t["status"] == "PREVISAO_BWEB_FORA_DA_AMOSTRA_1T_2026"
assert s["round"] == t["round"] == 1
assert s["source_year"] == t["year"] == 2026

def ler_csv_verificado(nome, sha_esperado, **kwargs):
    caminho = BASE / nome
    if not caminho.is_file():
        raise FileNotFoundError(f"Falta {caminho}. Execute S/T no menu.")
    real = sha256(caminho)
    if real != sha_esperado:
        raise ValueError(f"SHA-256 nao confere: {caminho.name}. Esperado {sha_esperado}; real {real}")
    return pd.read_csv(caminho, sep=";", encoding="utf-8-sig", **kwargs)

ea = ler_csv_verificado("phase5o_ea20_time_axes_2026.csv", s["ea20_timeline"]["sha256"])
bweb = ler_csv_verificado("phase5o_bweb_receipt_geo_2026.csv", s["bweb_geography"]["sha256"])
curva = ler_csv_verificado("phase5p_holdout_receipt_curve_2026.csv", t["output_files"]["post_cut_curve"]["sha256"])
ufs = ler_csv_verificado("phase5p_holdout_by_uf_2026.csv", t["output_files"]["per_UF"]["sha256"], dtype={"UF":str})
municipios = ler_csv_verificado("phase5p_holdout_by_municipality_2026.csv", t["output_files"]["per_municipality"]["sha256"], dtype={"UF":str, "municipio":str})

ea["data_hora"] = pd.to_datetime(ea["brt"])
bweb["data_hora"] = pd.to_datetime(bweb["brt_minute"])
curva["data_hora"] = pd.to_datetime(curva["brt_minute"])
print("Fontes verificadas com SHA-256: 5 CSV + relatorios de proveniencia S/T.")
print("Raiz:", ROOT)
print("EA20:", len(ea), "snapshots | BWEB:", len(bweb), "minutos | holdout:", len(curva), "minutos")


## 2 — Reconciliação e integridade básica

Certificar totais antes de interpretar qualquer gráfico. O teste T separa cada BU pelo **segundo exato**, embora o relatório agregue os lotes por minuto para visualização.

In [ ]:
totais = s["all_votes_reconciled"]
c = t["data"]
assert totais == {"sections": 499206, "valid": 119300788, "lula": 53879538, "flavio": 56104503}
assert int(ea.iloc[-1]["valid"]) == totais["valid"]
assert int(ea.iloc[-1]["lula"]) == totais["lula"]
assert int(ea.iloc[-1]["flavio"]) == totais["flavio"]
assert c["training_pre_cut_valid"] + c["holdout_post_cut_valid"] == totais["valid"]
assert c["holdout_post_cut_valid"] == int(curva.iloc[-1]["cum_post_valid"])
assert c["holdout_post_cut_lula"] == int(curva.iloc[-1]["cum_post_lula"])
assert c["holdout_post_cut_flavio"] == int(curva.iloc[-1]["cum_post_flavio"])
assert c["holdout_post_cut_valid"] == int(ufs["post_valid"].sum())
assert c["holdout_post_cut_valid"] == int(municipios["post_valid"].sum())
display(pd.DataFrame([
    {"Medida":"Seções BWEB", "Total":totais["sections"]},
    {"Medida":"Votos válidos BWEB/EA20", "Total":totais["valid"]},
    {"Medida":"Lula", "Total":totais["lula"]},
    {"Medida":"Flávio", "Total":totais["flavio"]},
    {"Medida":"Treino pré-corte BWEB", "Total":c["training_pre_cut_valid"]},
    {"Medida":"Teste pós-corte BWEB", "Total":c["holdout_post_cut_valid"]},
]))
print("OK: todos os somatorios nacionais e da cauda se reconciliam.")


## 3 — A suposta linha reta depende do eixo horizontal

Traçamos a mesma porcentagem acumulada de Lula no EA20 público sob três eixos distintos: **tempo físico**, **número ordinal de atualização** e **seções totalizadas**. O valor de R² mede somente ajuste descritivo de uma reta; não representa probabilidade de fraude ou regularidade. Os trechos entre capturas são segmentos desenhados, não observações contínuas.

In [ ]:
depois = ea.loc[ea["data_hora"] >= CORTE].copy().reset_index(drop=True)
depois["indice"] = range(len(depois))
eixos = [
    ("data_hora", "Horário real de publicação EA20", "clock"),
    ("indice", "Índice de atualização (espaçamento igual)", "updates"),
    ("st", "Seções totalizadas EA20", "sections"),
]
for coluna, titulo, chave in eixos:
    metr = s["ea20_post_resume_trajectory"]["share_lula_pct_" + chave]
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(depois[coluna], depois["share_lula_pct"], marker=".", markersize=3)
    ax.set_title(f"{titulo} | R² = {metr['r2']:.5f} | RMSE = {metr['rmse_pp']:.4f} pp")
    ax.set_ylabel("% acumulado de Lula (EA20)")
    ax.set_xlabel(titulo)
    ax.grid(alpha=0.25)
    fig.autofmt_xdate()
    plt.show()


## 4 — Recebimento de BUs é diferente de publicação EA20

Gráfico separado por horário de **recebimento**. Não sobreponha essa série à EA20 como se suas marcas de tempo descrevessem o mesmo evento. A diferença de volumes pós-corte significa *estágios distintos de processamento*, não votos desaparecidos.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(bweb["data_hora"], bweb["share_lula_pct"], label="BWEB recebido: Lula (%)")
ax.axvline(CORTE, linestyle="--", label="Corte 20:04:39 BRT")
ax.set(title="Curva cumulativa por DT_BU_RECEBIDO", xlabel="Horário do recebimento BWEB", ylabel="Lula (% dos válidos recebidos)")
ax.legend()
ax.grid(alpha=0.25)
fig.autofmt_xdate()
plt.show()
comparacao = t["EA20_versus_BWEB_different_events"]
display(pd.DataFrame([
    {"Estágio":"EA20 publicados depois do último snapshot no corte", "Válidos":comparacao["ea20_added_valid_to_final"]},
    {"Estágio":"BWEB recebidos em/apos o corte", "Válidos":comparacao["bweb_received_after_cutoff_valid"]},
    {"Estágio":"Diferença aritmética ENTRE ESTÁGIOS (não representa votos perdidos)", "Válidos":comparacao["non_comparable_difference_valid"]},
]))


## 5 — Fase T: previsão sem conhecer os votos futuros

Os perfis **BR pré-corte**, **UF pré-corte** e **município pré-corte** usam somente BUs recebidos antes de 20:04:39. O município pré-corte emprega suavização fixa de 1.000 votos da UF. `MUNICIPIO_final_IN_SAMPLE` usa o resultado final dos municípios, incluindo o próprio teste: é um **oráculo circular**, não uma previsão.

In [ ]:
f = c["forecast_models"]
nomes = {
    "BR_pre":"Brasil pré-corte",
    "UF_pre":"UF pré-corte",
    "MUNICIPIO_pre":"Município pré-corte",
    "MUNICIPIO_final_IN_SAMPLE":"Município final (oráculo)",
}
metricas = pd.DataFrame([
    {
        "Modelo": nomes[k],
        "Lula previsto (%)": v["pred_lula_pct"],
        "Lula real (%)": v["actual_lula_pct"],
        "Erro final Lula (pp)": v["lula_final_error_pp"],
        "MAE ponderado/min (pp)": v["weighted_minute_mae_lula_pp"],
        "RMSE ponderado/min (pp)": v["weighted_minute_rmse_lula_pp"],
    }
    for k, v in f.items()
])
display(metricas.round(4))
ax = metricas.plot.bar(x="Modelo", y="RMSE ponderado/min (pp)", figsize=(10, 4), legend=False)
ax.set(title="Erro por lote de minuto: modelo treinado vs oráculo", ylabel="RMSE ponderado (pp)", xlabel="")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()


## 6 — Trajetória dos votos da cauda: observado versus modelos

Aqui a abscissa é o **volume acumulado de votos BWEB pós-corte**, não o tempo de publicação EA20. A curva é acumulada: sua lisura pode ocultar grandes desvios entre lotes.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
series = [
    ("real_cum_post_lula_pct", "BWEB pós-corte observado"),
    ("br_pre_cum_lula_pct", "Brasil pré-corte"),
    ("uf_pre_cum_lula_pct", "UF pré-corte"),
    ("municipio_pre_cum_lula_pct", "Município pré-corte"),
    ("municipio_final_cum_lula_pct", "Oráculo municipal final (circular)"),
]
for coluna, legenda in series:
    ax.plot(curva["cum_post_valid"] / 1e6, curva[coluna], label=legenda)
ax.set(title="Participação de Lula entre os BUs da cauda", xlabel="Milhões de votos válidos BWEB recebidos após o corte", ylabel="Lula (% acumulado na cauda)")
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## 7 — Erro lote a lote, não só o agregado

O valor de erro `real_batch - municipio_pre_batch` mede, por minuto de recebimento, o desvio da composição real em relação à previsão feita com os municípios pré-corte. Minutos com poucos votos podem ser ruidosos; distinguimos explicitamente o subconjunto com pelo menos **5.000 válidos**.

In [ ]:
por_minuto = curva.loc[curva["d_valid"] >= 5000].copy()
por_minuto["erro_municipio_pp"] = por_minuto["real_batch_lula_pct"] - por_minuto["municipio_pre_batch_lula_pct"]
por_minuto["erro_uf_pp"] = por_minuto["real_batch_lula_pct"] - por_minuto["uf_pre_batch_lula_pct"]
fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(por_minuto["data_hora"], por_minuto["erro_municipio_pp"],
           s=10 + 100 * por_minuto["d_valid"] / por_minuto["d_valid"].max(),
           alpha=0.7, label="Município pré-corte")
ax.axhline(0, linestyle="--")
ax.set(title="Desvio por minuto da previsão municipal (> = 5.000 votos válidos)", xlabel="Horário BWEB", ylabel="Real menos previsto (pp)")
ax.grid(alpha=0.25)
fig.autofmt_xdate()
plt.show()
print("Minutos avaliados:", len(por_minuto), "| Votos válidos:", int(por_minuto["d_valid"].sum()))
display(por_minuto.nlargest(10, "erro_municipio_pp")[["brt_minute", "d_valid", "real_batch_lula_pct", "municipio_pre_batch_lula_pct", "erro_municipio_pp"]].round(3))


## 8 — Geografia dos resíduos: UFs

O código abaixo lista UFs que tiveram maiores diferenças entre votos pós-corte e a previsão **municipal pré-corte**, priorizando aquelas com volume material. O erro está definido como **previsto menos observado**, exatamente como na saída T.

In [ ]:
ufs_vol = ufs.loc[ufs["post_valid"] >= 10000].copy()
ufs_vol["erro_absoluto_pp"] = ufs_vol["error_lula_pp"].abs()
display(ufs_vol.sort_values("erro_absoluto_pp", ascending=False)[[
    "UF", "pre_valid", "post_valid", "post_lula_pct",
    "model_pre_lula_pct", "error_lula_pp",
    "unseen_post_valid", "unseen_post_pct"
]].head(20).round(3))
fig, ax = plt.subplots(figsize=(11, 4))
ax.scatter(ufs_vol["post_valid"], ufs_vol["error_lula_pp"])
for _, r in ufs_vol.iterrows():
    ax.annotate(r["UF"], (r["post_valid"], r["error_lula_pp"]), fontsize=8)
ax.axhline(0, linestyle="--")
ax.set(title="Previsão municipal pré-corte — resíduos por UF", xlabel="Votos válidos BWEB pós-corte", ylabel="Previsto menos observado (pp)")
ax.grid(alpha=0.25)
plt.show()


## 9 — Municípios: desvios grandes e municípios sem observação prévia

Os códigos de município são os **códigos utilizados no CSV**. A falta de votos antes do corte impede uma estimativa local independente; nesses casos o modelo usa o perfil da UF. Não interpretar o fallback como se fossem informações eleitorais conhecidas daquele município.

In [ ]:
m = municipios.copy()
m["erro_absoluto_pp"] = m["error_pp"].abs()
materiais = m.loc[m["post_valid"] >= 2000]
display(materiais.sort_values("erro_absoluto_pp", ascending=False)[[
    "UF", "municipio", "pre_valid", "post_valid",
    "observed_post_lula_pct", "predicted_post_lula_pct",
    "error_pp", "pre_available"
]].head(25).round(3))
n_sem = c["municipalities_without_pre_cut_voting"]
v_sem = c["post_votes_from_unseen_municipalities"]
print(f"Municípios sem votos pré-corte: {n_sem:,}")
print(f"Votos pós-corte desses municípios: {v_sem:,} ({100*v_sem/c['holdout_post_cut_valid']:.2f}% da cauda)")


## 10 — Distinção entre municípios com treino e sem treino

Uma parte dos erros pode ser causada por localidades inteiras recebidas tarde. Outra parte pode surgir de mudanças **dentro do mesmo município**, quando seções tardias têm perfil diferente das primeiras. O próximo quadro separa as duas situações; não decide, sozinho, se o mecanismo é administrativo, geográfico ou outra causa.

In [ ]:
m["grupo"] = m["pre_available"].map({True: "Com dados pré-corte", False: "Sem dados pré-corte"})
resumo = []
for grupo, df in m.groupby("grupo"):
    n = df["post_valid"].sum()
    mae = (df["post_valid"] * df["error_pp"].abs()).sum() / n if n else float("nan")
    vi = (df["post_valid"] * df["error_pp"]).sum() / n if n else float("nan")
    resumo.append({
        "Grupo":grupo, "Municípios":len(df),
        "Votos pós-corte":int(n), "Erro ponderado assinado (pp)":vi,
        "MAE ponderado (pp)":mae,
    })
display(pd.DataFrame(resumo).round(3))


## 11 — Auditoria da hipótese investigativa

**Achados que a Fase T permite avaliar:** diferença entre baseline Brasil, perfil por UF e por município; resíduos por minuto; origem geográfica dos maiores erros; participação dos municípios sem votos anteriores.

**O que a T não demonstra:** ela não reconstitui diretamente o mecanismo interno de publicação EA20, nem verifica um modelo de fila/buffering ou a ordem de inserção individual de votos no agregado publicado. O uso do resultado final por município no oráculo tem vazamento explícito.

**Próximas provas falsificáveis:** investigar as seções tardias *dentro dos municípios* com maior erro e testar modelos de atraso entre recebimento BWEB e publicação EA20 que preservem as contagens, a cronologia e os hashes. Não presumir fraude nem regularidade antes desses testes.

In [ ]:
print("REPRODUTIBILIDADE")
print("Repositorio:", REPO)
print("Raiz de dados:", ROOT)
print("Corte:", t["cutoff"])
print("Versao S:", s["version"])
print("Versao T:", t["version"])
print("SHA da serie EA20:", s["ea20_timeline"]["sha256"])
print("SHA da serie T:", t["output_files"]["post_cut_curve"]["sha256"])
print("Total de votos da cauda:", c["holdout_post_cut_valid"])
print("Primeiro turno 2026: sem download, ZIP, SQLite ou segundo turno.")
